# Reverse-Engineer a Vibe

### A suspiciously data-obsessed DJ has hired you.

Spotify has represented songs with numbers: *danceability*, *energy*, *valence*, *acousticness*, and more. Your job is to use those numbers to build a playlist for a very particular human situation—and then decide whether the numbers actually capture the vibe.

**Today you will practice:**

- loading and inspecting a dataframe
- selecting columns
- filtering rows with one or more conditions
- making a scatterplot
- using data to defend an interpretation

**The larger question:** What gets gained—and what gets lost—when we turn a human concept into numbers?

> **AI pause:** For this activity, keep Colab's AI/chat/code-generation tools closed. The goal is to practice translating an idea into pandas code yourself. You may use the syntax examples already in this notebook and ask a person for help.

## Mission map

1. Meet the data.
2. Build **dance-floor heartbreak** together.
3. Choose a route: **Guided Mission** or **Open Mission**.
4. Stress-test your playlist.
5. Prepare one recommendation—and one criticism of your own method—for the class showdown.

You can switch routes at any time. They are two kinds of support, not two different assignments.

## 1. Meet the data

Run the next cell. It downloads a public Spotify tracks dataset and prepares a classroom-friendly version. You do **not** need to understand every setup line yet.

The source contains roughly 114,000 tracks across many genres. For this activity, we keep tracks with a popularity score of at least 50 and remove repeated artist/title combinations. Popularity is a **snapshot from when the dataset was collected**, not a current chart position.

Dataset source: [Spotify Tracks Dataset on Hugging Face](https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset/resolve/main/dataset.csv"

raw_songs = pd.read_csv(DATA_URL)

# Classroom setup: keep useful columns, remove duplicate songs, and favor recognizable tracks.
songs = raw_songs[[
    "artists", "track_name", "popularity", "danceability", "energy",
    "acousticness", "instrumentalness", "speechiness", "valence",
    "tempo", "explicit", "track_genre"
]].drop_duplicates(subset=["artists", "track_name"])

songs = songs.rename(columns={"artists": "artist", "track_genre": "genre"})
songs = songs[songs["popularity"] >= 50].reset_index(drop=True)

print(f"Ready: {len(songs):,} songs")
songs.head()

### What do the numbers mean?

Most audio features range from **0 to 1**.

| Feature | Rough meaning |
|---|---|
| `valence` | musical positiveness: higher sounds more positive; lower sounds more negative |
| `energy` | intensity and activity |
| `danceability` | suitability for dancing |
| `acousticness` | confidence that the track is acoustic |
| `instrumentalness` | likelihood that the track has few or no vocals |
| `speechiness` | presence of spoken words |
| `popularity` | dataset snapshot on a 0–100 scale |
| `tempo` | estimated beats per minute |

Before running more code, discuss:

1. Which feature seems easiest to measure?
2. Which feature seems most interpretive?
3. What is missing if we want to describe a song's “vibe”?

In [ ]:
# Inspect the dataframe.
songs.info()

In [ ]:
# Select a few columns to make the table easier to read.
songs[["artist", "track_name", "valence", "energy", "danceability"]].head(10)

### Optional: search for an artist or song you know

Change the search words, but keep the quotation marks. This is a bonus tool; you do not need to memorize its syntax.

In [ ]:
search_words = "Taylor Swift"

songs[songs["artist"].str.contains(search_words, case=False, na=False)][[
    "artist", "track_name", "popularity", "valence", "energy", "danceability"
]].head(15)

## 2. Worked mission: Dance-floor heartbreak

We need songs that sound danceable but emotionally negative. That idea is still vague. To make it computable, we have to **operationalize** it:

> “Dance-floor heartbreak” = danceability above a chosen cutoff **and** valence below a chosen cutoff.

The cutoffs below are judgment calls, not facts. Run the cell, inspect the results, and then change at least one cutoff. Try to produce a list that is selective but not empty.

In [ ]:
# EDIT THESE TWO VALUES.
high_danceability = 0.75
low_valence = 0.35

heartbreak = songs[
    (songs["danceability"] > high_danceability) &
    (songs["valence"] < low_valence)
]

print(f"Your definition found {len(heartbreak):,} songs.")
heartbreak[[
    "artist", "track_name", "popularity", "danceability", "valence", "energy"
]].head(15)

**Checkpoint—answer in a text cell or your notes:**

- What did you count as “high” and “low”? Why?
- Do any results surprise you?
- When you change a cutoff, what happens to the number or quality of the results?
- Does low `valence` necessarily mean heartbreak?

### Coding checkpoint: rebuild the filter

Changing two numbers was only the warm-up. Now create the same playlist again **without copying the completed filter as a whole**.

In the next cell, type a pandas filter that:

1. starts with `songs[`;
2. includes one condition for `danceability`;
3. combines it with one condition for `valence` using `&`;
4. closes all parentheses and brackets;
5. saves the result as `heartbreak_rebuilt`.

Use the completed example above as a reference, but type the full expression yourself.

In [ ]:
# DELETE `None` AND TYPE YOUR COMPLETE FILTER.
heartbreak_rebuilt = None

# Then remove the # below to inspect your result.
# heartbreak_rebuilt[["artist", "track_name", "danceability", "valence"]].head(10)

## 3. Choose your route

Both routes have the same destination: define a vibe, use pandas to find candidates, and evaluate the result.

- Choose **Guided Mission** if you want a working structure to modify.
- Choose **Open Mission** if you would rather build the filter yourself.

### Route A: Guided Mission

Choose one prompt—or invent your own:

- rainy Sunday morning
- main-character entrance
- running away from your responsibilities
- villain entrance
- acoustic existential crisis
- aggressively wholesome
- beautiful menace

First, write a human definition in one sentence. Then decide which feature should be **high** and which should be **low**.

Use the prototype cell to test your idea quickly. Replace the four values with choices that fit your vibe. Keep feature names inside quotation marks.

In [ ]:
vibe_name = "rainy Sunday morning"

# Replace these features and cutoffs.
high_feature = "acousticness"
high_cutoff = 0.70

low_feature = "energy"
low_cutoff = 0.40

vibe = songs[
    (songs[high_feature] > high_cutoff) &
    (songs[low_feature] < low_cutoff)
]

print(f"{vibe_name}: {len(vibe):,} candidates")
vibe[[
    "artist", "track_name", "popularity",
    high_feature, low_feature
]].head(15)

If you get **too many** results, make a condition stricter. If you get **zero**, relax a cutoff. This is not merely debugging: your thresholds determine what your category means.

### Now replace the prototype with pandas code

The variables above helped you test the idea. Now write the dataframe filter directly. Do not use `high_feature`, `low_feature`, `high_cutoff`, or `low_cutoff` in your final version.

```python
vibe = songs[
    (songs["your first feature"] > your_cutoff) &
    (songs["your second feature"] < your_cutoff)
]
```

Type your version in the next cell and display the columns needed to judge it.

In [ ]:
# REPLACE `None` WITH YOUR COMPLETE TWO-CONDITION FILTER.
final_vibe = None

# Then display the evidence you need. Example structure:
# final_vibe[["artist", "track_name", "popularity", "energy", "valence"]].head(15)

**Guided extension:** Once the direct version works, add a third condition by copying this pattern into your filter:

```python
& (songs["popularity"] > 70)
```

### Route B: Open Mission

Invent a musical concept that Spotify does not explicitly measure. Then:

1. Give it a memorable name.
2. Define it in ordinary language.
3. Translate it into at least **three numerical conditions**.
4. Use pandas to find matching songs.
5. Create a second, competing definition of the same vibe and compare the results.

Write your own code below. You may copy and adapt the dance-floor heartbreak example.

In [ ]:
# OPEN MISSION WORKSPACE
# Write your own filter here. The comments keep the cell runnable until you begin.

# vibe_name = "..."
# vibe_a = songs[
#     (songs["..."] > ...) &
#     (songs["..."] < ...) &
#     (songs["..."] > ...)
# ]
# vibe_b = songs[ ...a genuinely different definition... ]
# Compare the two candidate lists.

### Optional debugging challenge

The DJ wrote the filter below using ordinary Python `and`. Run it, read the error, and repair it. Pandas conditions need `&`, with each comparison inside parentheses.

In [ ]:
# This is intentionally broken. Fix it.
# broken_vibe = songs[
#     songs["energy"] > 0.80 and songs["valence"] < 0.30
# ]

## 4. See your definition

A filter gives us a list. A scatterplot lets us see the rule as a region in the data.

If you used the Guided Mission, the cell works as written. If you used the Open Mission, set `x_feature` and `y_feature` to two features in your definition.

In [ ]:
# EDIT THESE if your vibe uses different features.
x_feature = high_feature
y_feature = low_feature

plt.figure(figsize=(9, 6))
plt.scatter(
    songs[x_feature], songs[y_feature],
    alpha=0.08, color="gray", label="all songs"
)
plt.scatter(
    vibe[x_feature], vibe[y_feature],
    alpha=0.45, color="#d1495b", label=vibe_name
)
plt.xlabel(x_feature)
plt.ylabel(y_feature)
plt.title(f"How we operationalized: {vibe_name}")
plt.legend()
plt.show()

### Read the picture

- Where did your filter carve out a region?
- Does the picture make your definition look sensible, arbitrary, or both?
- Would a different pair of features tell a different story?

**Optional experiment:** Change one axis to `popularity`, `valence`, or `danceability`. What becomes easier—or harder—to see?

## 5. Stress-test the algorithm

Choose **two or three candidates** from your results. Use songs you know, or listen to a short sample outside the notebook if time allows.

For each candidate, ask:

1. Does it actually fit the vibe?
2. Which number helped it qualify?
3. What human information did the filter miss—lyrics, cultural associations, genre, context, memory, irony?

Then find an **impostor**: a song that passes your numerical rule but does not feel right. An impostor is evidence about the limits of your definition, not a failed project.

In [ ]:
# Put one suspicious candidate's artist or title here.
candidate_words = ""

vibe[
    vibe["artist"].str.contains(candidate_words, case=False, na=False) |
    vibe["track_name"].str.contains(candidate_words, case=False, na=False)
][[
    "artist", "track_name", "popularity", "danceability",
    "energy", "acousticness", "valence", "tempo"
]].head(10)

## 6. Human vs. algorithm showdown

Prepare a 60-second pitch:

> **Our vibe:**  
> **Our human definition:**  
> **Our numerical definition:**  
> **One song the algorithm got right:**  
> **One impostor or limitation:**  
> **Our verdict:** Do these features capture the vibe, or only approximate it?

Class vote: **Would you let this algorithm control the aux cord?**

## If you finish early: Make the models compete

Create **two different numerical definitions** of the same vibe. For example:

- Version A: Sunday morning = high acousticness + low energy
- Version B: Sunday morning = high valence + low tempo

Compare their candidate lists or plots. Which operationalization is more convincing, and why?

This is the deeper data-science move: the dataframe does not choose the definition for you. You do.

---

### Data note

This classroom dataset is derived from the public [Spotify Tracks Dataset](https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset). Audio features and popularity values belong to the dataset snapshot and should not be treated as current measurements or as Spotify's complete account of a song.